# Blink Hardware Generalization Profiler

This notebook is designed to run on Google Colab to collect GPU profiling data for the Blink research paper. The collected data will include both model execution metrics and the underlying hardware specifications (e.g., TFLOPS, memory bandwidth), allowing Blink to extrapolate latency predictions across different GPU configurations.

## 1. Setup Environment

In [ ]:
!pip install torch torchvision pynvml thop pandas numpy

## 2. Hardware Specification Mapping
We embed known theoretical specs for common cloud GPUs (like T4, L4, V100, A100) based on NVIDIA's datasheets. If the connected GPU isn't listed, we fallback to querying the driver.

In [ ]:
import torch
import time
import numpy as np
import pynvml as nvml
import pandas as pd
import torchvision.models as models
import os
from datetime import datetime

# Known basic FP32 TFLOPS and memory bandwidth (GB/s) for Colab GPUs
GPU_SPECS = {
    "Tesla T4": {"tflops_fp32": 8.1, "memory_bandwidth_gbps": 320, "sm_count": 40},
    "NVIDIA L4": {"tflops_fp32": 30.3, "memory_bandwidth_gbps": 300, "sm_count": 58},
    "Tesla V100": {"tflops_fp32": 15.7, "memory_bandwidth_gbps": 900, "sm_count": 80},
    "Tesla P100": {"tflops_fp32": 9.3, "memory_bandwidth_gbps": 732, "sm_count": 56},
    "Tesla P100-PCIE-16GB": {"tflops_fp32": 9.3, "memory_bandwidth_gbps": 732, "sm_count": 56},
    "NVIDIA A100": {"tflops_fp32": 19.5, "memory_bandwidth_gbps": 1555, "sm_count": 108},
}

def get_hardware_features():
    if not torch.cuda.is_available():
        return {"gpu_name": "CPU", "tflops_fp32": 0.1, "memory_bandwidth_gbps": 50, "sm_count": 0}
        
    gpu_name = torch.cuda.get_device_name(0)
    basic_specs = {"gpu_name": gpu_name}
    
    # Match known Colab GPUs (partial match since names might have suffixes)
    matched = False
    for key, specs in GPU_SPECS.items():
        if key in gpu_name:
            basic_specs.update(specs)
            matched = True
            break
            
    if not matched:
        # Educated guess based on SM count if unknown
        sm_count = torch.cuda.get_device_properties(0).multi_processor_count
        basic_specs.update({
            "sm_count": sm_count,
            "tflops_fp32": sm_count * 0.2, # Very rough proxy
            "memory_bandwidth_gbps": sm_count * 8 # Very rough proxy
        })
        print(f"Warning: Unknown GPU {gpu_name}. Using roughly estimated hardware features.")
        
    return basic_specs

print("Detected Hardware Setup:")
hw_features = get_hardware_features()
for k, v in hw_features.items():
    print(f"{k}: {v}")

## 3. Redefine Profiling Logic
This adapts `model_profiler.py` to seamlessly append hardware parameters to every execution record.

In [ ]:
def profile_model_for_hardware(model, model_name, batch_sizes, hw_features):
    results = []
    device = torch.device("cuda")
    model = model.to(device)
    model.eval()
    
    # Base model stats
    total_params = sum(p.numel() for p in model.parameters())
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    model_size_mb = (param_size + buffer_size) / (1024 ** 2)
    
    for bs in batch_sizes:
        try:
            dummy_input = torch.randn(bs, 3, 224, 224, device=device)
            
            starter = torch.cuda.Event(enable_timing=True)
            ender = torch.cuda.Event(enable_timing=True)
            timings = []
            
            # Warmup
            with torch.no_grad():
                for _ in range(15):
                    _ = model(dummy_input)
            torch.cuda.synchronize()
            
            torch.cuda.reset_peak_memory_stats()
            
            # Measure
            with torch.no_grad():
                for _ in range(50):
                    starter.record()
                    _ = model(dummy_input)
                    ender.record()
                    torch.cuda.synchronize()
                    timings.append(starter.elapsed_time(ender))
                    
            peak_memory_mb = torch.cuda.max_memory_allocated() / (1024 ** 2)
            weight_memory_mb = param_size / (1024 ** 2)
            activation_memory_mb = max(0.0, peak_memory_mb - weight_memory_mb)
            
            record = {
                "model_name": model_name,
                "batch_size": bs,
                "total_parameters": total_params,
                "model_size_mb": model_size_mb,
                "execution_time_ms": float(np.median(timings)),
                "execution_time_std": float(np.std(timings)),
                "timing_cv": float(np.std(timings) / (np.median(timings) + 1e-9)),
                "peak_memory_mb": float(peak_memory_mb),
                "weight_memory_mb": float(weight_memory_mb),
                "activation_memory_mb": float(activation_memory_mb)
            }
            # Inject Hardware specs
            record.update(hw_features)
            
            results.append(record)
            print(f"{model_name} (BS={bs}): {record['execution_time_ms']:.2f} ms | {peak_memory_mb:.1f} MB")
            
        except torch.cuda.OutOfMemoryError:
            print(f"OOM at BS={bs} for {model_name}")
            torch.cuda.empty_cache()
            break
            
    return results

## 4. Run Standard Architectures
Generate the dataset across convolutional topologies.

In [ ]:
cnn_models = {
    "resnet18": models.resnet18(weights=None),
    "resnet50": models.resnet50(weights=None),
    "mobilenet_v2": models.mobilenet_v2(weights=None),
    "densenet121": models.densenet121(weights=None),
    "vgg16": models.vgg16(weights=None),
    "efficientnet_b0": models.efficientnet_b0(weights=None),
    "shufflenet_v2_x1_0": models.shufflenet_v2_x1_0(weights=None),
    "squeezenet1_0": models.squeezenet1_0(weights=None),
    "convnext_tiny": models.convnext_tiny(weights=None)
}

batch_sizes = [1, 2, 4, 8, 16, 32, 64, 128]
all_records = []

for name, m in cnn_models.items():
    print(f"\n--- Starting {name} ---")
    records = profile_model_for_hardware(m, name, batch_sizes, hw_features)
    all_records.extend(records)

# Save
df = pd.DataFrame(all_records)
os.makedirs("hw_data", exist_ok=True)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_name = f"hw_data/{hw_features['gpu_name'].replace(' ', '_')}_{timestamp}.csv"
df.to_csv(csv_name, index=False)
print(f"\nSaved generalization data to {csv_name}")
print("Please download this CSV and place it in the Neusight/data/raw/ folder of your local repo.")